In [13]:
!pip install supabase

In [14]:
import pandas as pd
import os
import numpy as np
from google.colab import drive
import math

# --- Configuracion ---
drive.mount('/content/drive')
from google.colab import userdata
from supabase import create_client

SUPABASE_URL = "https://ysbchbtuubrphiulkvlh.supabase.co"
SUPABASE_KEY = userdata.get('API_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
CSV_PATH = "/content/drive/MyDrive/CSV"
print("Listo")
# ---------------------

tables_order = [
    ('geolocation', 'olist_geolocation_dataset.csv'),
    ('product_category_name_translation', 'product_category_name_translation.csv'),
    ('customers', 'olist_customers_dataset.csv'),
    ('sellers', 'olist_sellers_dataset.csv'),
    ('products', 'olist_products_dataset.csv'),
    ('orders', 'olist_orders_dataset.csv'),
    ('order_items', 'olist_order_items_dataset.csv'),
    ('order_payments', 'olist_order_payments_dataset.csv'),
    ('order_reviews', 'olist_order_reviews_dataset.csv')
]

def cargar_datos_supabase():
    # Cargar traducciones para validación de FK
    df_cat = pd.read_csv(os.path.join(CSV_PATH, 'product_category_name_translation.csv'))
    categorias_validas = df_cat['product_category_name'].unique()

    def _sanitize_value(v):
        if v is None: return None
        if isinstance(v, (np.integer,)): return int(v)
        if isinstance(v, (np.floating,)):
            f = float(v)
            if math.isnan(f) or math.isinf(f): return None
            if f.is_integer(): return int(f)
            return f
        if isinstance(v, float):
            if math.isnan(v) or math.isinf(v): return None
            if v.is_integer(): return int(v)
            return v
        return v

    for table_name, file_name in tables_order:
        file_path = os.path.join(CSV_PATH, file_name)

        if os.path.exists(file_path):
            print(f"Iniciando carga de: {file_name}")
            df = pd.read_csv(file_path)

            # --- LIMPIEZA AGRESIVA PARA EVITAR ERRORES DE DUPLICADOS ---
            # Identificar posibles columnas de ID para eliminar duplicados reales del archivo
            id_cols = [c for c in df.columns if c.endswith('_id')]
            if id_cols:
                df = df.drop_duplicates(subset=id_cols, keep='last')

            # Limpieza de fechas
            for col in df.columns:
                if 'date' in col or 'timestamp' in col:
                    df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%S')

            df = df.replace([np.inf, -np.inf], np.nan)

            if table_name == 'products':
                df['product_category_name'] = df['product_category_name'].apply(
                    lambda x: x if x in categorias_validas else None
                )

            raw_data = df.to_dict(orient='records')

            # Carga por lotes
            batch_size = 500
            for i in range(0, len(raw_data), batch_size):
                batch_raw = raw_data[i:i + batch_size]
                batch = [{k: _sanitize_value(v) for k, v in rec.items()} for rec in batch_raw]

                try:
                    supabase.table(table_name).upsert(batch).execute()
                except Exception as e:
                    print(f"Error en lote {i} de {table_name}: {e}")

            print(f"Tabla {table_name} cargada correctamente.")

cargar_datos_supabase()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Listo
Iniciando carga de: olist_geolocation_dataset.csv
Tabla geolocation cargada correctamente.
Iniciando carga de: product_category_name_translation.csv
Tabla product_category_name_translation cargada correctamente.
Iniciando carga de: olist_customers_dataset.csv
Tabla customers cargada correctamente.
Iniciando carga de: olist_sellers_dataset.csv
Tabla sellers cargada correctamente.
Iniciando carga de: olist_products_dataset.csv
Tabla products cargada correctamente.
Iniciando carga de: olist_orders_dataset.csv
Tabla orders cargada correctamente.
Iniciando carga de: olist_order_items_dataset.csv
Tabla order_items cargada correctamente.
Iniciando carga de: olist_order_payments_dataset.csv
Tabla order_payments cargada correctamente.
Iniciando carga de: olist_order_reviews_dataset.csv
Error en lote 13000 de order_reviews: {'message': 'ON CONFLICT DO UPDATE comm